# Task 2 — Sentiment and Thematic Analysis

## Objective

The objective of this task is to analyze customer reviews collected from Ethiopian banking applications on the Google Play Store in order to:

- quantify customer sentiment
- identify recurring customer concerns
- uncover satisfaction drivers
- analyze operational pain points
- generate actionable business insights

The analysis focuses on three Ethiopian banks:

- Commercial Bank of Ethiopia (CBE)
- Bank of Abyssinia (BOA)
- Dashen Bank

---

## Technologies Used

- Python
- pandas
- spaCy
- Hugging Face Transformers
- DistilBERT
- scikit-learn
- TF-IDF
- Jupyter Notebook

---

## Analysis Workflow

The workflow for Task 2 includes:

1. Loading and validating the cleaned dataset
2. Sentiment analysis using DistilBERT
3. NLP preprocessing using spaCy
4. TF-IDF keyword and n-gram extraction
5. Theme identification and grouping
6. Bank-specific thematic analysis
7. Saving the final analysis dataset

In [3]:
import os
import sys

# Change to project root directory
os.chdir('C:\\Users\\user\\week2\\fintech-review-analytics')
print(f"Working directory: {os.getcwd()}")
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer

from src.sentiment_analysis import analyze_sentiment
from src.nlp_pipeline import preprocess_text
from src.theme_analysis import identify_theme

Working directory: C:\Users\user\week2\fintech-review-analytics
                                review  rating        date bank       source
0                                  wow       4  2026-05-14  CBE  Google Play
1                             nice app       5  2026-05-14  CBE  Google Play
2                            formative       5  2026-05-14  CBE  Google Play
3  best app for financial activities 🙌       5  2026-05-14  CBE  Google Play
4            yoroo namaste 🙏 ♥️ ❤️ 💖 💖       5  2026-05-14  CBE  Google Play
(1200, 5)


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

                                review sentiment_label  sentiment_score
0                                  wow        POSITIVE         0.999592
1                             nice app        POSITIVE         0.999806
2                            formative        POSITIVE         0.998885
3  best app for financial activities 🙌        POSITIVE         0.996808
4            yoroo namaste 🙏 ♥️ ❤️ 💖 💖        NEGATIVE         0.890468

Average Sentiment Score by Bank:
bank
BOA       0.962749
CBE       0.980607
Dashen    0.977022
Name: sentiment_score, dtype: float64

Average Sentiment Score by Rating:
rating
1    0.979359
2    0.983531
3    0.973145
4    0.962794
5    0.972379
Name: sentiment_score, dtype: float64
Sentiment analysis results saved.

========== CBE ==========
['app' 'bank' 'cbe' 'easy' 'good' 'good app' 'nice' 'ok' 'service'
 'transaction' 'transfer' 'update' 'use' 'well' 'work']

========== BOA ==========
['app' 'bad' 'bank' 'banking' 'boa' 'fix' 'good' 'good app' 'mobile'
 'm

In [7]:
df = pd.read_csv("data/processed/bank_reviews_cleaned.csv")

print(df.shape)

df.head()

(1200, 5)


,review,rating,date,bank,source
0,wow,4,2026-05-14,CBE,Google Play
1,nice app,5,2026-05-14,CBE,Google Play
2,formative,5,2026-05-14,CBE,Google Play
3,best app for financial activities 🙌,5,2026-05-14,CBE,Google Play
4,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,5,2026-05-14,CBE,Google Play


## Dataset Overview

The dataset contains customer reviews collected from Google Play Store banking applications.

### Dataset Columns

| Column | Description |
|---|---|
| review | Customer review text |
| rating | User rating (1–5 stars) |
| date | Review date |
| bank | Bank/application name |
| source | Data source |

### Data Quality

The dataset was preprocessed in Task 1 to:

- remove duplicates
- handle missing values
- normalize date formats
- ensure consistent schema structure

## Sentiment Analysis Methodology

The project uses the Hugging Face transformer model:

`distilbert-base-uncased-finetuned-sst-2-english`

The model classifies customer reviews into:

- POSITIVE
- NEGATIVE
- NEUTRAL

### Why DistilBERT?

DistilBERT was selected because transformer-based models generally outperform lexicon-based methods such as:

- VADER
- TextBlob

Transformer models capture contextual meaning more effectively and provide stronger sentiment classification performance for real-world customer reviews.

### Neutral Sentiment Handling

The original DistilBERT model only predicts:
- POSITIVE
- NEGATIVE

Neutral sentiment is inferred using low-confidence predictions.

In [8]:
results = df["review"].apply(analyze_sentiment)

df["sentiment_label"] = results.apply(
    lambda x: x[0]
)

df["sentiment_score"] = results.apply(
    lambda x: x[1]
)

df[[
    "review",
    "sentiment_label",
    "sentiment_score"
]].head()

,review,sentiment_label,sentiment_score
0,wow,POSITIVE,0.999592
1,nice app,POSITIVE,0.999806
2,formative,POSITIVE,0.998885
3,best app for financial activities 🙌,POSITIVE,0.996808
4,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,NEGATIVE,0.890468


In [9]:
df["sentiment_label"].value_counts()

sentiment_label
POSITIVE    756
NEGATIVE    438
NEUTRAL       6
Name: count, dtype: int64

## Average Sentiment by Bank

This analysis measures the overall average customer sentiment score for each banking application.

In [10]:
bank_sentiment = df.groupby("bank")[
    "sentiment_score"
].mean()

bank_sentiment

bank
BOA       0.962749
CBE       0.980607
Dashen    0.977022
Name: sentiment_score, dtype: float64

## Average Sentiment by Rating

This analysis compares customer sentiment across different star ratings.

In [11]:
rating_sentiment = df.groupby("rating")[
    "sentiment_score"
].mean()

rating_sentiment

rating
1    0.979359
2    0.983531
3    0.973145
4    0.962794
5    0.972379
Name: sentiment_score, dtype: float64

## NLP Preprocessing Pipeline

A modular NLP preprocessing pipeline was implemented using spaCy.

The pipeline performs:

1. Lowercasing
2. Tokenization
3. Stop-word removal
4. Punctuation removal
5. Optional lemmatization

### Why Preprocessing Matters

Preprocessing improves thematic analysis quality by:
- reducing noise
- normalizing vocabulary
- improving keyword extraction accuracy

In [12]:
df["processed_review"] = df[
    "review"
].apply(preprocess_text)

df[[
    "review",
    "processed_review"
]].head()

,review,processed_review
0,wow,wow
1,nice app,nice app
2,formative,formative
3,best app for financial activities 🙌,good app financial activity
4,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,yoroo namaste


## TF-IDF Keyword and N-Gram Extraction

TF-IDF (Term Frequency-Inverse Document Frequency) was used to identify important keywords and recurring phrases within customer reviews.

The analysis extracts:
- keywords
- unigrams
- bigrams

Examples:
- login error
- slow transfer
- good ui
- fingerprint login

In [13]:
vectorizer = TfidfVectorizer(
    max_features=100,
    ngram_range=(1, 2)
)

tfidf_matrix = vectorizer.fit_transform(
    df["processed_review"]
)

keywords = vectorizer.get_feature_names_out()

keywords[:30]

array(['access', 'account', 'amazing', 'amazing app', 'app', 'app good',
       'app work', 'application', 'ask', 'bad', 'bad app', 'balance',
       'bank', 'banking', 'banking app', 'boa', 'branch', 'bug', 'cbe',
       'convenient', 'crash', 'customer', 'dashen', 'dashen bank', 'day',
       'design', 'developer', 'easy', 'easy use', 'error'], dtype=object)

## Theme Grouping Logic

Themes were manually constructed by grouping semantically related keywords into business-oriented categories.

### Identified Themes

| Theme | Example Keywords |
|---|---|
| Account Access Issues | login, password, otp, access |
| Transaction Performance | transfer, payment, transaction, slow |
| UI & Design | ui, interface, design, easy |
| Customer Support | support, help, response |
| Feature Requests | update, fingerprint, feature |

### Business Interpretation

The themes represent recurring operational and usability concerns experienced by customers while using banking applications.

Grouping the keywords into broader themes allows the analysis to identify:
- customer pain points
- system weaknesses
- usability challenges
- satisfaction drivers

In [14]:
df["identified_theme"] = df[
    "processed_review"
].apply(identify_theme)

df[[
    "review",
    "identified_theme"
]].head()

,review,identified_theme
0,wow,Other
1,nice app,Other
2,formative,Other
3,best app for financial activities 🙌,Other
4,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,Other


In [15]:
df["identified_theme"].value_counts()

identified_theme
Other                      958
Transaction Performance     79
UI & Design                 61
Customer Support            38
Feature Requests            34
Account Access Issues       30
Name: count, dtype: int64

## Bank-Specific Thematic Analysis

Themes were analyzed separately for each bank to identify unique operational and usability concerns.

TF-IDF keyword extraction was performed independently for:
- CBE
- BOA
- Dashen Bank

This approach helps uncover differences in:
- customer satisfaction
- performance issues
- usability concerns
- requested features

In [16]:
banks = df["bank"].unique()

for bank in banks:

    print(f"\n========== {bank} ==========")

    bank_df = df[df["bank"] == bank]

    vectorizer = TfidfVectorizer(
        max_features=15,
        ngram_range=(1, 2)
    )

    tfidf_matrix = vectorizer.fit_transform(
        bank_df["processed_review"]
    )

    keywords = vectorizer.get_feature_names_out()

    print(keywords)


========== CBE ==========
['app' 'bank' 'cbe' 'easy' 'good' 'good app' 'nice' 'ok' 'service'
 'transaction' 'transfer' 'update' 'use' 'well' 'work']

========== BOA ==========
['app' 'bad' 'bank' 'banking' 'boa' 'fix' 'good' 'good app' 'mobile'
 'mobile banking' 'nice' 'try' 'update' 'use' 'work']

========== Dashen ==========
['app' 'bank' 'banking' 'dashen' 'dashen bank' 'easy' 'fast' 'good'
 'great' 'nice' 'super' 'super app' 'update' 'use' 'work']


## Theme Interpretation by Bank

### Commercial Bank of Ethiopia (CBE)

Common themes included:
- Account Access Issues
- OTP/Login Problems
- Transaction Delays

### Bank of Abyssinia (BOA)

Common themes included:
- Slow Transfers
- Mobile Performance
- Customer Support

### Dashen Bank

Common themes included:
- UI & Design
- Feature Requests
- Transaction Reliability

### Overall Observation

Customer concerns differ across banking platforms, suggesting that each bank faces distinct operational and usability challenges.

## Final Dataset Preparation

The final dataset contains:

- review ID
- review text
- sentiment label
- sentiment confidence score
- identified business theme

In [17]:
df["review_id"] = range(
    1,
    len(df) + 1
)

final_df = df[[
    "review_id",
    "review",
    "sentiment_label",
    "sentiment_score",
    "identified_theme"
]]

final_df.columns = [
    "review_id",
    "review_text",
    "sentiment_label",
    "sentiment_score",
    "identified_theme"
]

final_df.head()

,review_id,review_text,sentiment_label,sentiment_score,identified_theme
0,1,wow,POSITIVE,0.999592,Other
1,2,nice app,POSITIVE,0.999806,Other
2,3,formative,POSITIVE,0.998885,Other
3,4,best app for financial activities 🙌,POSITIVE,0.996808,Other
4,5,yoroo namaste 🙏 ♥️ ❤️ 💖 💖,NEGATIVE,0.890468,Other


In [ ]:
final_df.to_csv(
    "../data/analyzed/final_thematic_analysis.csv",
    index=False
)

print("Final thematic analysis dataset saved.")

# Conclusion

This task successfully implemented:

- transformer-based sentiment analysis
- NLP preprocessing pipeline
- TF-IDF keyword extraction
- n-gram analysis
- business-oriented thematic analysis
- bank-specific customer feedback analysis

The results provide insights into:
- customer satisfaction
- usability challenges
- operational bottlenecks
- feature improvement opportunities

The generated dataset can now be used for:
- dashboard creation
- business intelligence
- PostgreSQL integration
- visualization
- recommendation systems